# Image Dataset and DataLoader Setup

This notebook contains the necessary imports, data downloads, transforms, and DataLoaders copied from `pytorch_custom_dataset.ipynb` to prepare the image data for a new model.

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from PIL import Image
import requests
import zipfile
from pathlib import Path
import os

# Device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
# Setup path to a datafolder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder does not exist, download and prepare it
if image_path.is_dir():
    print(f"{image_path} Directory already exists... skipping download")
else:
    print(f"{image_path} does not exist creating one")
    image_path.mkdir(parents=True, exist_ok=True)
    
    # Download the dataset
    zip_filepath = data_path / "pizza_steak_sushi.zip"
    with open(zip_filepath, "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/refs/heads/main/data/pizza_steak_sushi.zip")
        print("Downloading pizza steak sushi data...")
        f.write(request.content)
        
    # Unzip pizza steak sushi file
    with zipfile.ZipFile(zip_filepath, "r") as zip_ref:
        print("Unzipping pizza steak sushi data...")
        zip_ref.extractall(image_path)

data\pizza_steak_sushi Directory already exists... skipping download


In [3]:
# Setup train and test paths
train_dir = image_path / "train"
test_dir = image_path / "test"
train_dir, test_dir

(WindowsPath('data/pizza_steak_sushi/train'),
 WindowsPath('data/pizza_steak_sushi/test'))

In [4]:
# Write a transform for the images
data_transform = transforms.Compose([
    # Resize our images to 64 * 64
    transforms.Resize(size=(64, 64)),
    # Flip the images randomly on the horizontal
    transforms.RandomHorizontalFlip(p=0.5),
    # Convert to Tensor
    transforms.ToTensor()
])

In [5]:
# Turn train and test datasets into ImageFolder datasets
train_data = datasets.ImageFolder(root=train_dir, transform=data_transform, target_transform=None)
test_data = datasets.ImageFolder(root=test_dir, transform=data_transform, target_transform=None)

class_names = train_data.classes
class_dict = train_data.class_to_idx

print(f"Train data length: {len(train_data)}")
print(f"Test data length: {len(test_data)}")
print(f"Class Names: {class_names}")

Train data length: 225
Test data length: 75
Class Names: ['pizza', 'steak', 'sushi']


In [6]:
# Turn datasets into DataLoaders
BATCH_SIZE = 32
NUM_WORKERS = 0  # 0 is recommended for compatibility on Windows

train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False
)

print(f"Train Dataloader batches: {len(train_dataloader)}")
print(f"Test Dataloader batches: {len(test_dataloader)}")

Train Dataloader batches: 8
Test Dataloader batches: 3


In [7]:
# Retrieve a single batch to verify shapes
img_batch, label_batch = next(iter(train_dataloader))
print(f"Image batch shape: {img_batch.shape} -> [batch_size, color_channels, height, width]")
print(f"Label batch shape: {label_batch.shape}")

Image batch shape: torch.Size([32, 3, 64, 64]) -> [batch_size, color_channels, height, width]
Label batch shape: torch.Size([32])


## Custom Dataset and Custom DataLoader Setup

Below, we define a custom subclass of `torch.utils.data.Dataset` called `ImageFolderCustom` to replicate the functionality of `torchvision.datasets.ImageFolder` manually, along with its corresponding DataLoaders.

In [8]:
def find_classes(directory: str) -> tuple[list[str], dict[str, int]]:
    """
    Finds the class folder names in a target directory.
    """
    # 1. Get the class names by scanning the target directory
    classes = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())
    
    # 2. Raise an error if class names could not be found
    if not classes:
        raise FileNotFoundError(f"Could not find any classes in the {directory}\nPlease check file structure")
        
    # 3. Create a dictionary of index labels
    class_to_idx = {class_name: i for i, class_name in enumerate(classes)}
    return classes, class_to_idx

In [9]:
class ImageFolderCustom(Dataset):
    def __init__(self, target_dir: str, transform=None):
        self.paths = list(Path(target_dir).glob("*/*.jpg"))
        # Setup transforms
        self.transform = transform
        # Create classes and class_to_idx attributes
        self.classes, self.class_to_idx = find_classes(target_dir)

    # Create a function to load images
    def load_image(self, index: int) -> Image.Image:
        """
        Opens an image via a path and returns it
        """
        image_path = self.paths[index]
        return Image.open(image_path)

    # Overwrite __len__()
    def __len__(self) -> int:
        "Returns the total number of samples"
        return len(self.paths)

    # Overwrite __getitem__() method to return a particular sample
    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        "Returns one sample of data, data and label (X and Y)"
        img = self.load_image(index)
        class_name = self.paths[index].parent.name  # Expects path in format: data_folder/class_name/image.jpg
        class_idx = self.class_to_idx[class_name]
        
        # Transform if necessary
        if self.transform:
            return self.transform(img), class_idx
        else:
            return img, class_idx  # Return untransformed image and label

In [10]:
# Create transforms for Custom Dataset
train_transforms_custom = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
])

test_transforms_custom = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.ToTensor()
])

In [11]:
# Create instances of ImageFolderCustom
train_data_custom = ImageFolderCustom(target_dir=train_dir, transform=train_transforms_custom)
test_data_custom = ImageFolderCustom(target_dir=test_dir, transform=test_transforms_custom)

print(f"Custom Train data classes: {train_data_custom.classes}")
print(f"Custom Train data class to idx: {train_data_custom.class_to_idx}")

Custom Train data classes: ['pizza', 'steak', 'sushi']
Custom Train data class to idx: {'pizza': 0, 'steak': 1, 'sushi': 2}


In [12]:
# Create Dataloaders from Custom Dataset instances
BATCH_SIZE = 32
NUM_WORKERS = 0

train_dataloader_custom = DataLoader(
    dataset=train_data_custom,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True
)

test_dataloader_custom = DataLoader(
    dataset=test_data_custom,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False
)

train_dataloader_custom, test_dataloader_custom

(<torch.utils.data.dataloader.DataLoader at 0x1b0ebcdae60>,
 <torch.utils.data.dataloader.DataLoader at 0x1b0ebc3f5e0>)

In [13]:
# Retrieve a single batch to verify shapes from Custom DataLoader
img_custom_batch, label_custom_batch = next(iter(train_dataloader_custom))
print(f"Custom Image batch shape: {img_custom_batch.shape} -> [batch_size, color_channels, height, width]")
print(f"Custom Label batch shape: {label_custom_batch.shape}")

Custom Image batch shape: torch.Size([32, 3, 64, 64]) -> [batch_size, color_channels, height, width]
Custom Label batch shape: torch.Size([32])


### 7.1 Creating a cnn of TinyVGG architecture

In [14]:
class TinyVGG(nn.Module):
    """
        Model architecture copying TinyVgg architecture from CNN explainer
    """
    def __init__(self,input_shape:int,hidden_units:int,output_shape:int) -> None:
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,out_channels=hidden_units,
            kernel_size=3,stride= 1 , padding= 1),
            
            nn.ReLU(),
            
            nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,
            kernel_size=3, stride=1 ,padding=1   ),

            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
            

        self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,
        kernel_size=3,stride= 1 , padding= 1 ),
        
        nn.ReLU(),
        
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,
        kernel_size=3, stride=1 ,padding=1 ),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units*16*16,out_features=output_shape    )
        )
    def forward(self,x):
        x = self.conv_block_1(x)
        #print(x.shape)
        x = self.conv_block_2(x)
        #print(x.shape)
        x = self.classifier(x)
        #print(x.shape)
        return x
        #return self.classifier(self.conv_block_2(self.conv_block_1(x)))
        #https://horace.io/brrr_intro.html

In [15]:
torch.manual_seed(42)
model_0 = TinyVGG(input_shape=3,hidden_units=10,output_shape= len(class_names)).to(device)

model_0


TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2560, out_features=3, bias=True)
  )
)

### 7.3 Try a forward pass using a single image

In [16]:
from torchvision import transforms

# 1. Create a transform pipeline for the training data
train_transforms = transforms.Compose([
    transforms.Resize(size=(64, 64)), # Resize images to 64x64
    transforms.ToTensor()             # Convert PIL images to PyTorch Tensors (and scale pixels to 0-1)
])

# 2. Create a transform pipeline for the testing data
test_transforms = transforms.Compose([
    transforms.Resize(size=(64, 64)), # Resize images to 64x64
    transforms.ToTensor()             # Convert PIL images to PyTorch Tensors
])

print(f"Train transforms:\n{train_transforms}")
print(f"Test transforms:\n{test_transforms}")

Train transforms:
Compose(
    Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
)
Test transforms:
Compose(
    Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
)


In [17]:
from torchvision import datasets
from torch.utils.data import DataLoader

# 1. Create the "Simple" Datasets using PyTorch's built-in ImageFolder
train_data_simple = datasets.ImageFolder(root=train_dir, 
                                         transform=train_transforms)

test_data_simple = datasets.ImageFolder(root=test_dir, 
                                        transform=test_transforms)

# 2. Setup batch size and number of workers
BATCH_SIZE = 32
NUM_WORKERS = 0 # Keeps it safe for Windows Jupyter execution

# 3. Create the DataLoaders
train_dataloader_simple = DataLoader(dataset=train_data_simple,
                                     batch_size=BATCH_SIZE,
                                     num_workers=NUM_WORKERS,
                                     shuffle=True)

test_dataloader_simple = DataLoader(dataset=test_data_simple,
                                    batch_size=BATCH_SIZE,
                                    num_workers=NUM_WORKERS,
                                    shuffle=False)

print(f"Train dataloader: {train_dataloader_simple}")
print(f"Test dataloader: {test_dataloader_simple}")

Train dataloader: <torch.utils.data.dataloader.DataLoader object at 0x000001B0EBD1B6A0>
Test dataloader: <torch.utils.data.dataloader.DataLoader object at 0x000001B0EBD1BD90>


In [18]:
image_batch , label_batch = next(iter(train_dataloader_simple))
image_batch.shape,label_batch.shape
image_batch,label_batch = image_batch.to(device) , label_batch.to(device)


In [19]:
#Try a forward pass
model_0(image_batch)

tensor([[0.0578, 0.0634, 0.0351],
        [0.0657, 0.0650, 0.0398],
        [0.0702, 0.0678, 0.0412],
        [0.0687, 0.0646, 0.0417],
        [0.0625, 0.0613, 0.0355],
        [0.0587, 0.0595, 0.0358],
        [0.0634, 0.0637, 0.0376],
        [0.0653, 0.0615, 0.0376],
        [0.0678, 0.0621, 0.0349],
        [0.0620, 0.0599, 0.0371],
        [0.0607, 0.0628, 0.0387],
        [0.0638, 0.0589, 0.0352],
        [0.0658, 0.0611, 0.0369],
        [0.0703, 0.0689, 0.0330],
        [0.0557, 0.0571, 0.0356],
        [0.0632, 0.0642, 0.0388],
        [0.0639, 0.0616, 0.0375],
        [0.0605, 0.0600, 0.0374],
        [0.0623, 0.0627, 0.0368],
        [0.0740, 0.0676, 0.0400],
        [0.0621, 0.0613, 0.0361],
        [0.0632, 0.0599, 0.0366],
        [0.0662, 0.0627, 0.0334],
        [0.0637, 0.0622, 0.0394],
        [0.0670, 0.0666, 0.0363],
        [0.0650, 0.0601, 0.0405],
        [0.0639, 0.0597, 0.0395],
        [0.0733, 0.0687, 0.0412],
        [0.0697, 0.0631, 0.0402],
        [0.062

In [20]:
10*16*16


2560

### 7.4 Using `torchinfo` to get an idea of the shapes going through our model

In [21]:
from torchinfo import summary
summary(model_0,input_size= [1,3,64,64])

Layer (type:depth-idx)                   Output Shape              Param #
TinyVGG                                  [1, 3]                    --
├─Sequential: 1-1                        [1, 10, 32, 32]           --
│    └─Conv2d: 2-1                       [1, 10, 64, 64]           280
│    └─ReLU: 2-2                         [1, 10, 64, 64]           --
│    └─Conv2d: 2-3                       [1, 10, 64, 64]           910
│    └─ReLU: 2-4                         [1, 10, 64, 64]           --
│    └─MaxPool2d: 2-5                    [1, 10, 32, 32]           --
├─Sequential: 1-2                        [1, 10, 16, 16]           --
│    └─Conv2d: 2-6                       [1, 10, 32, 32]           910
│    └─ReLU: 2-7                         [1, 10, 32, 32]           --
│    └─Conv2d: 2-8                       [1, 10, 32, 32]           910
│    └─ReLU: 2-9                         [1, 10, 32, 32]           --
│    └─MaxPool2d: 2-10                   [1, 10, 16, 16]           --
├─Sequentia

### 7.5 Create train and test loops functions

`train_step()` - Takes in a model and dataloader and trains the model on the dataloader

`test_step()` - Takes in a model and a dataloader and tests the model on the dataloader


In [22]:
from train_test import test_step , train_step,eval_model
from helper_function import accuracy_fn,print_train_time
from tqdm.auto import tqdm

#Setting up loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params= model_0.parameters(),lr = 0.1)



In [23]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [24]:
device

'cuda'

In [28]:

from tqdm.auto import tqdm
torch.manual_seed(42)
from helper_function import print_train_time
from timeit import default_timer as timer 

def train_step(model:torch.nn.Module,dataloader : torch.utils.data.DataLoader,
                loss_fn  : torch.nn.Module ,optimizer : torch.optim.Optimizer,device = device):
    #Put the model in train mode
    model_0.train()
    #Setup the train loss and train accuracy values
    train_loss,train_acc= 0,0
    
    for batch,(X,y) in enumerate(dataloader):
        #Send the data to the target data
        X,y = X.to(device),y.to(device)
        #Forward pass
        y_pred = model_0(X)
        #Calculate the loss 
        loss = loss_fn(y_pred,y)
        train_loss += loss.item()
        

        optimizer.zero_grad()

        loss.backward()
        
        optimizer.step()

        #Calculate the accumulated accuracy metric
        y_pred_class = torch.argmax(torch.softmax(y_pred,dim=1),dim=1)
        train_acc = (y_pred_class == y ).sum().item() / len(y_pred)

    #Adjust metrics to get avrage loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    






In [29]:
#Create test step
def test_step(model:torch.nn.Module,dataloader : torch.utils.data.DataLoader,
                loss_fn  : torch.nn.Module, device = device):
    #put model in eval mode
    model.eval()

    #Setup test loss and test accuracy values
    test_loss,test_acc = 0,0
    #Turn on inference mode
    with torch.inference_mode():
        #Loop through dataloader batches
        for batch,(X,y) in enumerate(dataloader):
            #Send the data to target device
            X ,y = X.to(device) , y.to(device)

            #Forward pass
            test_pred_logits = model(X)

            #Calcualte the loss
            loss = loss_fn(test_pred_logits,y)
            test_loss += loss.item()

            #Calculte the accruacy
            test_pred_labels = test_pred_logits.argmax(dim = 1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))

    #Adjust metircs to get average loss and accuracy per batch
    test_loss = test_loss / len(data_loader)
    test_acc = test_acc / len(data_loader)
    return test_loss,test_acc
    



### 7.6 Creating a `train()` function to combine `train_step()` and `test_step()`


In [30]:
from tqdm.auto import tqdm

#1.Create a train function that takes in various model parameters + optimizer + dataloaders + loss function
def train(model : torch.nn.Module,train_dataloader : torch.utils.data.DataLoader,test_dataloader : torch.utils.data.DataLoader,
        optimizer:torch.optim.Optimizer,loss_fn : torch.nn.Module = nn.CrossEntropyLoss(),epochs:int = 5,device = device ):
    #2. Create empty results dictionary 
    results = {'train_loss':[],
                'train_acc':[],
                'test_loss':[],
                'test_acc':[]}
    for epoch in tqdm(range(epochs)):
        train_loss , train_acc = train_step(model=model,
                                            dataloader = train_dataloader,
                                            loss_fn = loss_fn,
                                            optimizer = optimizer,
                                            device = device)